In [1]:
import pandas as pd
import re

# File paths
input_file = "ALL AHN SHIPPED SPECIMENS.xlsx"
output_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"

# Read the Excel file
df = pd.read_excel(input_file, dtype=str)

# Remove rows with missing Barcode
df = df[df["Barcode"].notna()]

# Function to generate *follow_ups.submitter_id
def generate_followup_id(row):
    barcode = row["Barcode"]
    study_visit = str(row["Study Visit"]).strip()
    
    if not isinstance(barcode, str) or len(barcode) < 5:
        return ""

    patient_id = barcode[:5]

    if study_visit.startswith("Week"):
        match = re.search(r"Week\s*(\d+)", study_visit)
        if match:
            week_num = int(match.group(1))
            day_equivalent = week_num * 7
            return f"{patient_id}_obs_{day_equivalent}"
    elif study_visit.startswith("Day"):
        match = re.search(r"Day\s*(\d+)", study_visit)
        if match:
            day_num = int(match.group(1))
            return f"{patient_id}_clinical_{day_num}"
    elif study_visit == "Baseline":
        return f"{patient_id}_clinical_0"
    
    return ""

# Apply follow-up ID generation
df["*follow_ups.submitter_id"] = df.apply(generate_followup_id, axis=1)

# Standardize labs.submitter_id
lab_mapping = {
    "CCF": "Cleveland Clinic",
    "IU": "Indiana University",
    "Mayo": "Mayo Rochester",
    "ULouisville": "University of Louisville",
    "UPitt": "University of Pittsburgh",
    "UTSW": "University of Texas Southwestern",
    "VCU": "Virginia Commonwealth University"
}
# First apply the initial mapping
df["labs.submitter_id"] = df["Clinical Site of origin"].replace(lab_mapping)

# Then refine UMMS/BIDMC using Barcode rules
def resolve_umms_bidmc(row):
    site = row["Clinical Site of origin"]
    barcode = str(row["Barcode"])
    
    if site == "UMMS/BIDMC":
        if barcode.startswith("41"):
            return "University of Massachusetts"
        elif barcode.startswith("42"):
            return "Beth Israel Deaconess Medical Center"
    return row["labs.submitter_id"]

df["labs.submitter_id"] = df.apply(resolve_umms_bidmc, axis=1)

# Standardize specimen_type
specimen_mapping = {
    "CPT plasma": "CPT Plasma",
    "HCl sodium citrate plasma": "HCl Sodium Citrate Plasma",
    "NEAT plasma": "NEAT Plasma",
    "Platelet-poor plasma": "Platelet Poor Plasma",
    "Platelet-rich plasma": "Platelet Rich Plasma",
    "Whole blood": "Whole Blood (DNA)"
}
df["specimen_type"] = df["Specimen Type"].replace(specimen_mapping)

# Build the output DataFrame
df_output = pd.DataFrame()
df_output["*type"] = ["aliquot"] * len(df)
df_output["project_id"] = ["ARDaC-AlcHepNet"] * len(df)
df_output["*submitter_id"] = df["Barcode"]
df_output["*follow_ups.submitter_id"] = df["*follow_ups.submitter_id"]
df_output["labs.submitter_id"] = df["labs.submitter_id"]
df_output["aliquot_amount"] = ""
df_output["aliquot_collection_unit"] = ""
df_output["container_type"] = ""
df_output["specimen_type"] = df["specimen_type"]

# Final mapping for labs.submitter_id to standardized lab IDs
final_lab_id_mapping = {
    "Not Assigned": "lab_0",
    "Beth Israel Deaconess Medical Center": "lab_1",
    "Cleveland Clinic": "lab_2",
    "Indiana University": "lab_3",
    "Mayo Clinic Arizona": "lab_4",
    "University of Massachusetts": "lab_5",
    "University of Louisville": "lab_6",
    "University of Pittsburgh": "lab_7",
    "University of Texas Southwestern": "lab_8",
    "UT Southwestern - Parkland": "lab_9",
    "Virginia Commonwealth University": "lab_10",
    "Mayo Clinic Florida": "lab_11",
    "Mayo Clinic Rochester": "lab_12"
}

# Apply the final mapping
df_output["labs.submitter_id"] = df_output["labs.submitter_id"].replace(final_lab_id_mapping)
# Save as TSV
df_output.to_csv(output_file, sep="\t", index=False)